# DeepLabV3 Water Mask Segmentation - Interactive Notebook

This notebook provides an interactive environment for exploring water body segmentation from satellite imagery using DeepLabV3.

## Table of Contents
1. [Setup and Imports](#setup)
2. [Data Loading and Preprocessing](#data-loading)
3. [Model Architecture](#model)
4. [Inference and Visualization](#inference)
5. [Results Analysis](#analysis)
6. [Interactive Demo](#demo)

## 1. Setup and Imports <a id="setup"></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
import pandas as pd
from skimage import measure, morphology
import seaborn as sns
from ipywidgets import interact, widgets, Layout
from IPython.display import display, HTML
import warnings
warnings.filterwarnings('ignore')

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ All libraries imported successfully!")
print(f"📊 PyTorch version: {torch.__version__}")
print(f"🖼️ OpenCV version: {cv2.__version__}")

## 2. Data Loading and Preprocessing <a id="data-loading"></a>

In [ ]:
class ImageProcessor:
    """
    Image preprocessing utilities for satellite imagery
    """
    
    def __init__(self, target_size=(512, 512)):
        self.target_size = target_size
        
    def load_image(self, image_path):
        """Load image from path"""
        image = Image.open(image_path).convert('RGB')
        return np.array(image)
    
    def resize_image(self, image, size=None):
        """Resize image to target size"""
        if size is None:
            size = self.target_size
        return cv2.resize(image, size)
    
    def normalize_image(self, image):
        """Normalize image to [0, 1] range"""
        return image.astype(np.float32) / 255.0
    
    def apply_clahe(self, image):
        """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization)"""
        lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        lab[:,:,0] = clahe.apply(lab[:,:,0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
    
    def preprocess_pipeline(self, image, apply_clahe=False, normalize=True):
        """Complete preprocessing pipeline"""
        # Resize
        processed = self.resize_image(image)
        
        # Apply CLAHE if requested
        if apply_clahe:
            processed = self.apply_clahe(processed)
        
        # Normalize
        if normalize:
            processed = self.normalize_image(processed)
            
        return processed

# Initialize processor
processor = ImageProcessor()
print("✅ Image processor initialized!")

## 3. Model Architecture <a id="model"></a>

Here we define the DeepLabV3 model architecture for water segmentation.

In [ ]:
class WaterSegmentationModel:
    """
    DeepLabV3 model wrapper for water segmentation
    """
    
    def __init__(self, model_type='deeplabv3_resnet50', num_classes=2, pretrained=True):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_type = model_type
        self.num_classes = num_classes
        
        # Load pretrained model
        if model_type == 'deeplabv3_resnet50':
            self.model = models.segmentation.deeplabv3_resnet50(pretrained=pretrained)
        elif model_type == 'deeplabv3_resnet101':
            self.model = models.segmentation.deeplabv3_resnet101(pretrained=pretrained)
        else:
            raise ValueError(f"Unknown model type: {model_type}")
        
        # Modify classifier for binary segmentation
        self.model.classifier[4] = nn.Conv2d(256, num_classes, kernel_size=1)
        
        self.model.to(self.device)
        self.model.eval()
        
        # Define transforms
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                               std=[0.229, 0.224, 0.225])
        ])
        
    def preprocess_input(self, image):
        """Preprocess image for model input"""
        if isinstance(image, np.ndarray):
            image = Image.fromarray((image * 255).astype(np.uint8) if image.max() <= 1 else image.astype(np.uint8))
        
        tensor = self.transform(image).unsqueeze(0)
        return tensor.to(self.device)
    
    def predict(self, image):
        """Predict water mask for input image"""
        input_tensor = self.preprocess_input(image)
        
        with torch.no_grad():
            output = self.model(input_tensor)['out']
            probabilities = torch.softmax(output, dim=1)
            prediction = torch.argmax(probabilities, dim=1)
            
        return prediction.cpu().numpy()[0], probabilities.cpu().numpy()[0]
    
    def mock_predict(self, image):
        """Mock prediction for demonstration (replace with actual model)"""
        # Convert to grayscale and apply threshold
        if len(image.shape) == 3:
            gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        else:
            gray = (image * 255).astype(np.uint8)
            
        # Apply Gaussian blur and threshold
        blurred = cv2.GaussianBlur(gray, (5, 5), 0)
        _, binary = cv2.threshold(blurred, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        
        # Create probability map
        probs = np.stack([binary, 1 - binary], axis=0)
        
        return 1 - binary, probs

# Initialize model (using mock for demo)
print(f"🔧 Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")
model = WaterSegmentationModel()
print("✅ Model initialized!")

## 4. Inference and Visualization <a id="inference"></a>

In [ ]:
class ResultVisualizer:
    """
    Visualization utilities for segmentation results
    """
    
    @staticmethod
    def plot_results(original, mask, overlay=None, figsize=(15, 5)):
        """Plot original image, mask, and overlay"""
        fig, axes = plt.subplots(1, 3 if overlay is not None else 2, figsize=figsize)
        
        # Original image
        axes[0].imshow(original)
        axes[0].set_title('Original Image', fontsize=14)
        axes[0].axis('off')
        
        # Mask
        axes[1].imshow(mask, cmap='Blues')
        axes[1].set_title('Water Mask', fontsize=14)
        axes[1].axis('off')
        
        # Overlay if provided
        if overlay is not None:
            axes[2].imshow(overlay)
            axes[2].set_title('Overlay', fontsize=14)
            axes[2].axis('off')
        
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def create_overlay(original, mask, alpha=0.3, water_color=[0, 100, 255]):
        """Create overlay visualization"""
        overlay = original.copy()
        if original.max() <= 1:
            overlay = (overlay * 255).astype(np.uint8)
        
        # Create colored mask
        colored_mask = np.zeros_like(overlay)
        colored_mask[mask == 1] = water_color
        
        # Blend images
        result = cv2.addWeighted(overlay, 1-alpha, colored_mask, alpha, 0)
        return result
    
    @staticmethod
    def analyze_segmentation(mask):
        """Analyze segmentation results"""
        total_pixels = mask.size
        water_pixels = np.sum(mask == 1)
        water_percentage = (water_pixels / total_pixels) * 100
        
        # Connected components analysis
        labeled_mask = measure.label(mask)
        props = measure.regionprops(labeled_mask)
        
        results = {
            'total_pixels': total_pixels,
            'water_pixels': water_pixels,
            'water_percentage': water_percentage,
            'num_water_bodies': len(props),
            'largest_water_body': max([p.area for p in props]) if props else 0
        }
        
        return results

visualizer = ResultVisualizer()
print("✅ Visualizer initialized!")

## 5. Results Analysis <a id="analysis"></a>

In [ ]:
def create_sample_image():
    """Create a sample satellite-like image for demonstration"""
    # Create a synthetic satellite image
    height, width = 512, 512
    
    # Base terrain (brown/green)
    image = np.random.rand(height, width, 3) * 0.3
    image[:, :, 1] += 0.3  # Add green channel
    image[:, :, 0] += 0.2  # Add red channel
    
    # Add water bodies (dark blue areas)
    # River
    cv2.rectangle(image, (100, 200), (400, 220), (0.1, 0.1, 0.4), -1)
    
    # Lake
    cv2.circle(image, (300, 350), 60, (0.05, 0.05, 0.3), -1)
    
    # Small pond
    cv2.circle(image, (150, 100), 25, (0.08, 0.08, 0.35), -1)
    
    # Add some noise for realism
    noise = np.random.normal(0, 0.05, image.shape)
    image = np.clip(image + noise, 0, 1)
    
    return image

# Create sample image
sample_image = create_sample_image()
plt.figure(figsize=(8, 8))
plt.imshow(sample_image)
plt.title('Sample Satellite Image', fontsize=16)
plt.axis('off')
plt.show()

# Process with model
processed_image = processor.preprocess_pipeline(sample_image, apply_clahe=False, normalize=True)
mask, probabilities = model.mock_predict(processed_image)

# Create visualizations
overlay = visualizer.create_overlay(sample_image, mask)
visualizer.plot_results(sample_image, mask, overlay)

# Analyze results
analysis = visualizer.analyze_segmentation(mask)
print("\n📊 Segmentation Analysis:")
for key, value in analysis.items():
    print(f"  {key}: {value}")

## 6. Interactive Demo <a id="demo"></a>

Use the interactive widgets below to experiment with different parameters.

In [ ]:
def interactive_segmentation(apply_clahe=False, confidence_threshold=0.5, overlay_alpha=0.3):
    """Interactive function for parameter tuning"""
    # Process image
    processed = processor.preprocess_pipeline(sample_image, apply_clahe=apply_clahe, normalize=True)
    mask, probs = model.mock_predict(processed)
    
    # Apply confidence threshold
    confident_mask = (probs[1] > confidence_threshold).astype(int)
    
    # Create overlay
    overlay = visualizer.create_overlay(sample_image, confident_mask, alpha=overlay_alpha)
    
    # Visualize
    visualizer.plot_results(sample_image, confident_mask, overlay, figsize=(18, 6))
    
    # Analysis
    analysis = visualizer.analyze_segmentation(confident_mask)
    print(f"\n📊 Water Coverage: {analysis['water_percentage']:.2f}%")
    print(f"🏞️ Number of Water Bodies: {analysis['num_water_bodies']}")

# Create interactive widgets
interact(interactive_segmentation,
         apply_clahe=widgets.Checkbox(value=False, description='Apply CLAHE'),
         confidence_threshold=widgets.FloatSlider(value=0.5, min=0.1, max=0.9, step=0.1, description='Confidence'),
         overlay_alpha=widgets.FloatSlider(value=0.3, min=0.1, max=0.8, step=0.1, description='Overlay Alpha'));

## Upload Your Own Image

Use the file upload widget below to test with your own satellite images.

In [ ]:
from ipywidgets import FileUpload, Output, VBox, HBox, Button
from IPython.display import clear_output

# Create upload widget
upload_widget = FileUpload(
    accept='.jpg,.jpeg,.png,.tiff,.tif',  # Accepted file extension
    multiple=False,  # Allow only one file
    description='Upload Image',
    layout=Layout(width='300px')
)

process_button = Button(
    description='Process Uploaded Image',
    button_style='primary',
    layout=Layout(width='200px')
)

output_widget = Output()

def process_uploaded_image(button):
    with output_widget:
        clear_output(wait=True)
        
        if not upload_widget.value:
            print("❌ Please upload an image first!")
            return
        
        try:
            # Get uploaded file
            uploaded_file = list(upload_widget.value.values())[0]
            
            # Load image
            image = Image.open(io.BytesIO(uploaded_file['content']))
            image_array = np.array(image.convert('RGB'))
            
            print(f"✅ Loaded image: {image_array.shape}")
            
            # Process image
            processed = processor.preprocess_pipeline(image_array, normalize=True)
            mask, probs = model.mock_predict(processed)
            
            # Create overlay
            overlay = visualizer.create_overlay(processed, mask)
            
            # Visualize
            visualizer.plot_results(processed, mask, overlay)
            
            # Analysis
            analysis = visualizer.analyze_segmentation(mask)
            print(f"\n📊 Analysis Results:")
            for key, value in analysis.items():
                print(f"  {key}: {value}")
                
        except Exception as e:
            print(f"❌ Error processing image: {str(e)}")

process_button.on_click(process_uploaded_image)

# Display widgets
display(VBox([
    HBox([upload_widget, process_button]),
    output_widget
]))

## Performance Metrics

Compare different segmentation approaches and analyze performance.

In [ ]:
def compare_methods():
    """Compare different segmentation methods"""
    methods = {
        'Otsu Threshold': lambda img: cv2.threshold(cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY), 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1],
        'Adaptive Threshold': lambda img: cv2.adaptiveThreshold(cv2.cvtColor((img*255).astype(np.uint8), cv2.COLOR_RGB2GRAY), 1, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 11, 2),
        'Mock DeepLabV3': lambda img: model.mock_predict(img)[0]
    }
    
    fig, axes = plt.subplots(2, len(methods), figsize=(15, 8))
    
    processed_sample = processor.preprocess_pipeline(sample_image)
    
    for i, (method_name, method_func) in enumerate(methods.items()):
        # Apply method
        result = method_func(processed_sample)
        
        # Invert if needed (ensure water is 1)
        if method_name != 'Mock DeepLabV3':
            result = 1 - result
        
        # Plot mask
        axes[0, i].imshow(result, cmap='Blues')
        axes[0, i].set_title(f'{method_name}\nMask', fontsize=12)
        axes[0, i].axis('off')
        
        # Plot overlay
        overlay = visualizer.create_overlay(sample_image, result)
        axes[1, i].imshow(overlay)
        axes[1, i].set_title(f'{method_name}\nOverlay', fontsize=12)
        axes[1, i].axis('off')
        
        # Calculate metrics
        analysis = visualizer.analyze_segmentation(result)
        print(f"\n{method_name}:")
        print(f"  Water Coverage: {analysis['water_percentage']:.2f}%")
        print(f"  Water Bodies: {analysis['num_water_bodies']}")
    
    plt.tight_layout()
    plt.show()

compare_methods()

## Next Steps

To enhance this notebook with a real DeepLabV3 model:

1. **Train the Model**: Use your satellite imagery dataset to train a DeepLabV3 model
2. **Load Pretrained Weights**: Replace the mock model with actual trained weights
3. **Fine-tune Parameters**: Adjust preprocessing and postprocessing based on your data
4. **Add Evaluation Metrics**: Implement IoU, F1-score, and other segmentation metrics
5. **Batch Processing**: Add support for processing multiple images

## Conclusion

This interactive notebook provides a complete framework for water body segmentation from satellite imagery. The modular design allows for easy integration of trained models and extension with additional features.